# Comment Classification: LLM"

Before running this notebook:
1. Install Ollama: https://ollama.com
2. Pull the model: ollama pull qwen3:8b

In [13]:
# importing libraries

import pandas as pd
import ollama
import json
import re
from tqdm import tqdm

In [14]:
# loading comments data
data_cpath = "data/comments/comments.csv"
data_comments = pd.read_csv(data_cpath, encoding="latin1")

# loading evaluation data
data_epath = "data/comments/comments_manual_coding.xlsx"
data_eval = pd.read_excel(data_epath, engine="openpyxl")


In [27]:
data_eval["breadth"] = data_eval["breadth"].clip(upper=3)

In [15]:
# defining the system prompt inline with the codebook

SYSTEM_PROMPT = """You are coding German-language user comments for a social science study
titled "Likes or Dislikes, Gratifications or Concerns?", about a German 9-point anti-terrorism
plan / terrorism policy. The unit of analysis is a single user comment. Code the comment as a
whole, independently, on the six variables below. Do not infer beyond what the text states.

===========================================================
GENERAL CODING RULES (apply to all six variables)
===========================================================
- Code each comment as a whole and independently on all six dimensions.
- If a comment is unclear, code the dominant impression.
- Short reply comments that ONLY agree with another comment (e.g. "sehe ich auch so" / "I see
  it the same way", "stimmt", "genau meine Meinung") are NOT coded with automatic default
  values. If a parent comment is provided below, and the reply clearly just endorses it, code
  pol_opin and breadth by reference to the content being endorsed. However, such replies are
  NEVER coded as emotional expression (emot_exp = 0), even if the parent comment was emotional.
  Never infer pers_exp from the parent - only code it 1 if the reply itself describes a
  personal experience.
- Very short comments (1-2 words) that stand alone (no parent) should be coded based on their
  apparent sentiment and content, not defaulted to 0/absent.
- For breadth, count the number of distinct topics addressed using the topic list below; each
  topic is counted once regardless of how much of the comment it takes up.
- For controversiality, the key question is whether the CONTENT is likely to provoke strong
  disagreement among other users - regardless of the commenter's own intent.

===========================================================
1. pers_exp (0/1) - Personal experience
===========================================================
Key question: does the comment reveal something about the commenter's own life, not just
their opinion about the topic?

0 (Absent): Comment does not include any personal biographical information or intimate
experiences. No reference to the commenter's own life events.
Examples:
- "Weapons purchases should not only be restricted online." (Waffenkauf sollte nicht nur
  online eingeschraenkt werden.)
- "Deploying the armed forces internally violates the Basic Law." (Bundeswehr im Inneren?
  Hier verstoesst sie gegen das Grundgesetz.)

1 (Present): Comment includes a personal biographical experience, a reference to a specific
event the commenter lived through, or intimate personal circumstances (including close family
members).
Examples:
- "I witnessed the terrorist attacks in Turkey last year." (Ich habe letztes Jahr die
  Terroranschlaege in der Tuerkei miterlebt.)
- "My brother was near the attack at the time." (Mein Bruder war damals in der Naehe des
  Anschlags.)

===========================================================
2. emot_exp (0/1) - Emotional expression
===========================================================
0 (Absent): Comment does not contain emotional language or affective expressions. Tone is
purely factual, analytical, or opinion-based without explicit emotion words.
Examples:
- "Hire more staff." (Mehr Personal einstellen.)
- "Of course big promises are made that are never kept after the vote." (Natuerlich werden
  grosse Versprechen gemacht, die nach der Wahl nie eingehalten werden.)

1 (Present): Comment contains explicit emotion/feeling words (e.g. Angst, Freude, gut,
schlecht, ich fuehle, ich bin froh, ich bin besorgt) OR the comment is overall more
emotional/expressive than evaluative.
Examples:
- "I have become afraid and changed my behavior." (Ich habe Angst und mein Verhalten
  geaendert.)
- "I find it unsettling that so little is happening." (Ich finde es beunruhigend, dass so
  wenig passiert.)

===========================================================
3. pol_opin (0/1) - Political opinion
===========================================================
0 (Absent): Comment does not express a political stance, party preference, ideological
position, or evaluation of political actors or policies of the commenter's own. This includes
regular evaluations of the suggested plan itself or neutral ideas for implementation.
Examples:
- "Definitely better cooperation between international intelligence services." (Unbedingt
  eine bessere Zusammenarbeit der internationalen Geheimdienste.)
- "Hire more staff." (Mehr Personal einstellen.)

1 (Present): Comment expresses a political opinion, evaluates a politician or party, takes an
ideological stance, or advocates for a specific course of action that is clearly political.
Examples:
- "Hot air, like everything that comes from Merkel." (Heisse Luft, wie Alles, was von Merkel
  kommt.)
- "The deportation of refugees must be intensified." (Eine Rueckfuehrung von Fluechtlingen
  muss verstaerkt werden.)

===========================================================
4. breadth (0-3) - Number of distinct topics
===========================================================
Topic list (count each distinct topic once, cap total at 3): politicians'/political
institutions' failure, migration/refugees/integration, religion, security measures, civil
liberties, international cooperation, injustice/suffering, terrorism/radicalization,
democracy (civic engagement, participation, elections, education, parties,
politicians/candidates, etc.)

0 = No topics: comment addresses none of the listed topics (a general reaction/evaluation
without a specific topic).
Example: "Ineffective gibberish." (Effektlose Gelaber.) - general reaction, no distinct topic.

1 = One topic: comment stays focused on a single topic or aspect of the plan.
Examples:
- "Hire more staff." (Mehr Personal einstellen.) - security measures
- "Weapons purchases should not only be restricted online." (Waffenkauf sollte nicht nur
  online eingeschraenkt werden.) - security measures

2 = Two topics: comment addresses two distinct topics from the list.

3 = Three (or more): comment spans three or more distinct topics.
Example: "We need better intelligence services, more police, and a clear deportation policy."
(Wir brauchen bessere Geheimdienste, mehr Polizei und eine klare Abschiebepolitik.) -
international cooperation + security measures + migration.

===========================================================
5. valence (1-4) - Sentiment toward the topic
===========================================================
1 = Positive: comment expresses overwhelmingly positive sentiment.
Examples:
- "The best Chancellor has initiated concrete measures." (Die beste Bundeskanzlerin hat
  konkrete Massnahmen angeschoben.)
- "I find the 9-point plan sensible for better protecting the population." (Ich finde den
  9-Punkte-Plan sinnvoll, um die Bevoelkerung mehr zu schuetzen.)
- "It is a good start! Potential threats must be expelled." (Es ist ein guter Anfang!
  Potenzielle Gefaehrder muessen ausgewiesen werden.)

2 = Negative: comment expresses overwhelmingly negative sentiment.
Examples:
- "Hot air, like everything that comes from Merkel." (Heisse Luft, wie Alles, was von Merkel
  kommt.)
- "Nine points of blabla, nothing concrete, a bit of waffle." (Neun Punkte BlaBla, nichts
  konkretes, ein bisschen Gesuelze.)
- "Ineffective gibberish." (Effektlose Gelaber.)

3 = Ambivalent: comment expresses both positive and negative sentiment, typically with
explicit contrast markers ("aber", "jedoch", "einerseits...andererseits" / but, however, on
one hand...on the other hand).
Examples:
- "In principle the plan is a good start, but I lack faith in the concrete implementation."
  (Im Prinzip ist der Plan ein guter Anfang, allein mir fehlt der Glaube an der Umsetzung.)
- "Going in the right direction, but the plan alone is not enough." (Geht in die richtige
  Richtung - aber der Plan allein reicht nicht.)
- "Reads quite well, but overall far too vague." (Liest sich ganz gut, ist aber insgesamt
  viel zu unkonkret.)

4 = Indifferent: comment expresses little to no evaluative sentiment - short agreements,
neutral observations, or replies that merely echo another comment without adding a stance.
Examples:
- "I see it the same way." (Sehe ich auch so.)
- "Agreed." (Einverstanden.)
- "I fully agree with the statement." (Der Aussage stimme ich voll und ganz zu.)

===========================================================
6. contr (0/1) - Controversiality
===========================================================
0 (Not controversial): comment is largely factual, procedural, or broadly consensual;
unlikely to provoke strong disagreement or negative reactions from other users.
Examples:
- "Definitely better cooperation between international intelligence services." (Unbedingt
  eine bessere Zusammenarbeit der internationalen Geheimdienste.)
- "Hire more staff." (mehr personal einstellen)
- "Weapons purchases should not only be restricted online." (Waffenkauf sollte nicht nur
  online eingeschraenkt werden.)

1 (Controversial): comment touches predominantly on highly divisive political or social
topics likely to provoke strong disagreement, such as immigration, religion, ethnicity, or
radical political positions.
Examples:
- "Why not stop the intake of asylum seekers. That saves a lot of costs." (Warum nicht Schluss
  mit den Aufnahmen. Das spart viel Kosten.)
- "Hot air, like everything that comes from Merkel." (Heisse Luft, wie Alles, was von Merkel
  kommt.)
- "The deportation of refugees must be intensified." (Eine Rueckfuehrung von Fluechtlingen
  muss verstaerkt werden.)

===========================================================
Output format
===========================================================
Return ONLY a single-line JSON object with exactly these six integer keys, nothing else, no
explanation, no markdown code fences:
{"pers_exp": 0, "emot_exp": 0, "pol_opin": 0, "breadth": 0, "valence": 1, "contr": 0}
"""

In [16]:
# define classifier function with qwen3:8b set as default

def classify_comment(text, parent_text=None, model="qwen3:8b"):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ],
        options={"temperature": 0}
    )

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [17]:
# post_number resets per topic, so it's not unique on its own - build a composite key instead
def make_id(topic, post_number):
    return f"{topic}_{int(post_number)}"

# build ids on the FULL data_comments first, so parent lookups still work even for
# replies whose parent falls outside whatever subset we classify below
data_comments["row_id"] = data_comments.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
data_eval["row_id"] = data_eval.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
id_col = "row_id"

reply_col = "reply_to_post_number" if "reply_to_post_number" in data_comments.columns else None
text_by_id = dict(zip(data_comments["row_id"], data_comments["raw"].fillna("")))

def get_parent_text(row):
    if reply_col and pd.notna(row.get(reply_col)):
        parent_id = make_id(row["topic"], row[reply_col])
        return text_by_id.get(parent_id)
    return None

data_comments_full = data_comments.copy()  # keep the unfiltered version for the full run later

# for now, only classify comments that already have a manual code (validation pass) -
# comment this line out later to run the full dataset instead
data_comments = data_comments[data_comments[id_col].isin(data_eval[id_col])].copy()
print(f"Classifying {len(data_comments)} comments")

Classifying 150 comments


In [18]:
MODELS = ["qwen3:8b", "llama3.1:8b"]

In [19]:
VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
texts = data_comments["raw"].fillna("")


# delete:
#data_comments = data_comments.head().copy()


for model_name in MODELS:
    results = []
    for i, row in tqdm(data_comments.iterrows(), total=len(data_comments), desc=f"Classifying ({model_name})"):
        parent_text = get_parent_text(row)
        codes = classify_comment(texts.loc[i], parent_text=parent_text, model=model_name)
        results.append(codes)

    results_df = pd.DataFrame(results)
    model_tag = model_name.replace(":", "_").replace(".", "_")
    pred_cols = [f"{v}_{model_tag}" for v in VARS]
    data_comments[pred_cols] = results_df.values

Classifying (llama3.1:8b): 100%|██████████| 150/150 [29:18<00:00, 11.72s/it]


In [20]:
data_comments.head()

,coder,post_number,user,topic,raw,pers_exp,emot_exp,pol_opin,breadth,valence,...,pol_opin_qwen3_8b,breadth_qwen3_8b,valence_qwen3_8b,contr_qwen3_8b,pers_exp_llama3_1_8b,emot_exp_llama3_1_8b,pol_opin_llama3_1_8b,breadth_llama3_1_8b,valence_llama3_1_8b,contr_llama3_1_8b
12,NaN,19,Maksimo,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Ich stimme dem 9 Punkteplan zu in allen Bereic...,NaN,NaN,NaN,NaN,NaN,...,1,2,1,1,0,1,1,2,2,1
27,NaN,34,brunolina,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Spontan - ohne jetzt auf die einzelnen Inhalte...,NaN,NaN,NaN,NaN,NaN,...,1,1,2,1,1,1,1,2,2,0
29,NaN,36,holzwurmpaul,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Überfällig ja aber wer hat soviel Eier in der ...,NaN,NaN,NaN,NaN,NaN,...,1,1,2,1,1,1,1,3,2,1
49,NaN,56,Sudiko,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Also ich finde auch das die Bundeswehr im Inne...,NaN,NaN,NaN,NaN,NaN,...,1,2,3,1,0,1,1,3,2,1
57,NaN,64,Ewald,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Vom Ansatz her richtig. Auch eine Einbindung d...,NaN,NaN,NaN,NaN,NaN,...,1,3,3,1,0,1,1,3,2,1


In [33]:
import numpy as np
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
var_weights = {"pers_exp": None, "emot_exp": None, "pol_opin": None, "contr": None,
               "breadth": "linear", "valence": None}
var_levels = {"pers_exp": "nominal", "emot_exp": "nominal", "pol_opin": "nominal",
              "contr": "nominal", "breadth": "interval", "valence": "nominal"}

def krippendorff_alpha(y_true, y_pred, level):
    try:
        data = np.array([y_true, y_pred], dtype=float)
        return krippendorff.alpha(reliability_data=data, level_of_measurement=level)
    except (ZeroDivisionError, ValueError):
        return float("nan")

data_eval_renamed = data_eval.rename(columns={v: f"{v}_true" for v in VARS})
merged = data_eval_renamed.merge(data_comments, on=id_col)
print(f"Matched {len(merged)} of {len(data_eval)} rows\n")

all_rows = []

for model_name in MODELS:
    model_tag = model_name.replace(":", "_").replace(".", "_")
    print(f"\n########## {model_name} ##########")
    for var in VARS:
        y_true = merged[f"{var}_true"]
        y_pred = merged[f"{var}_{model_tag}"]

        mask = y_true.notna() & y_pred.notna() & (y_pred != -1)
        y_true_m, y_pred_m = y_true[mask], y_pred[mask]

        print(f"=== {var} (n={mask.sum()}) ===")
        report_dict = classification_report(y_true_m, y_pred_m, zero_division=0, output_dict=True)
        print(classification_report(y_true_m, y_pred_m, zero_division=0))

        kappa = cohen_kappa_score(y_true_m, y_pred_m, weights=var_weights[var])
        alpha = krippendorff_alpha(y_true_m.values, y_pred_m.values, var_levels[var])
        print(f"Cohen's Kappa: {kappa:.3f}   Krippendorff's alpha ({var_levels[var]}): {alpha:.3f}\n")

        for class_label, metrics in report_dict.items():
            if isinstance(metrics, dict):  # skips "accuracy", which is a bare float
                all_rows.append({
                    "model": model_name, "variable": var, "n": mask.sum(),
                    "class": class_label,
                    "precision": metrics["precision"], "recall": metrics["recall"],
                    "f1_score": metrics["f1-score"], "support": metrics["support"],
                    "accuracy": round(report_dict["accuracy"], 3),
                    "cohens_kappa": round(kappa, 3),
                    "krippendorff_alpha": round(alpha, 3) if not np.isnan(alpha) else "n/a",
                })

report_df = pd.DataFrame(all_rows)
report_df.to_excel("data/eval_report.xlsx", index=False)
print("Saved eval_report.xlsx")

Matched 150 of 150 rows


########## qwen3:8b ##########
=== pers_exp (n=150) ===
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       134
           1       0.75      0.75      0.75        16

    accuracy                           0.95       150
   macro avg       0.86      0.86      0.86       150
weighted avg       0.95      0.95      0.95       150

Cohen's Kappa: 0.720   Krippendorff's alpha (nominal): 0.721

=== emot_exp (n=150) ===
              precision    recall  f1-score   support

           0       0.90      0.85      0.88       108
           1       0.67      0.76      0.71        42

    accuracy                           0.83       150
   macro avg       0.78      0.81      0.79       150
weighted avg       0.84      0.83      0.83       150

Cohen's Kappa: 0.588   Krippendorff's alpha (nominal): 0.589

=== pol_opin (n=150) ===
              precision    recall  f1-score   support

           0       1.00      0.27  

In [12]:
# full run - classify every comment, not just the validation subset

FINAL_MODEL = "qwen3:8b"

texts_full = data_comments_full["raw"].fillna("")

results_full = []
for i, row in tqdm(data_comments_full.iterrows(), total=len(data_comments_full), desc=f"Classifying (full, {FINAL_MODEL})"):
    parent_text = get_parent_text(row)
    codes = classify_comment(texts_full.loc[i], parent_text=parent_text, model=FINAL_MODEL)
    results_full.append(codes)

Classifying (full, qwen3:8b):   0%|          | 0/1189 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
results_full_df = pd.DataFrame(results_full)
data_comments_full[["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]] = results_full_df.values
data_comments_full[["raw", "pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]].head()

In [ ]:
data_comments_full.to_excel("data_comments_classified_full.xlsx", index=False)